# Dealing with gradients

Sometimes we have data not only for the variable of interest, but also its gradient. This is a common situation when working with potential fields for implicit geological modelling.

In [ ]:
%%capture
!pip install cmcrameri # scientific color maps
!pip install git+https://github.com/italo-goncalves/geoML.git@claude

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cmcrameri import cm

import geoml

import geoml.kernels as kr
import geoml.transform as tr
import geoml.latent as gl

## The dataset

The dataset is included in the package. It contains data points and a set of directions representing tangents to the potential field.

In [ ]:
points, tangents, _ = geoml.datasets.example_fold()

print('Point data:')
print(points)
print("\nTangents:")
print(tangents)

Location map:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=[9, 9])
ax.scatter(points.coordinates[:, 0], points.coordinates[:, 1],
           c=points.variables["rock_num"].measurements.values,
           cmap=cm.roma, vmin=-1, vmax=1)
ax.set_aspect("equal")
ax.set_xlabel("X")
ax.set_ylabel("Y")

for i in range(tangents.n_data):
    ax.arrow(
        tangents.coordinates[i, 0], tangents.coordinates[i, 1],
        tangents.directions[i, 0]*2, tangents.directions[i, 1]*2
    )
    ax.arrow(
        tangents.coordinates[i, 0], tangents.coordinates[i, 1],
        -tangents.directions[i, 0]*2, -tangents.directions[i, 1]*2
    )

fig.show()

## Implicit modelling with directional data

The VGP supports data in the form of gradients. The `tangents` data object is assumed to represent directions with no variation. It will be used to constrain the potential field.

Note that each category must have its own potential field but, as we only have two categories, their field differ only in sign.

In [ ]:
cov = kr.Covariance(
    kernel=kr.Cubic(),
    transform=tr.Isotropic(50)
)

# Inducing points
ip_grid = geoml.data.Grid2D(start=[0,0], n=[21, 21], end=[100, 100])

# Latent variable network
net_in = gl.GradientConstrainedInput(
    inducing_points=ip_grid,
    directional_data=tangents,
    covariance=cov,
    size=1
    )
net_out = gl.Linear(net_in, size=2) # to match the 2 categories

# net_in = gl.BasicInput(
#     inducing_points=ip_grid,
#     transform=tr.Isotropic(50),
#     fix_transform=True
#     )
# gp = gl.BasicGP(net_in, kernel=kr.Cubic())
# net_out = gl.Linear(gp, size=2) # to match the 2 categories

# The model
model = geoml.models.VGPNetwork(
    data=points,
    variables='rock',
    likelihoods=geoml.likelihood.CategoricalGaussianIndicator(2),
    latent_network=net_out
    )
model.train_full(1000)

### Prediction and results

Let us create a grid to receive the predictions.

In [ ]:
grid = geoml.data.Grid2D(start=[0, 0], n=[501, 501], step=[0.2, 0.2])

model.predict(grid)

In [ ]:
pred_potential = grid.variables["rock"].components['a'].indicator_predicted.as_image()

max_potential = np.floor(np.max(pred_potential))
min_potential = np.ceil(np.min(pred_potential))
c_amp = np.maximum(max_potential, -min_potential)
pot_steps = np.linspace(min_potential, max_potential, 50)

uncertainty = grid.variables["rock"].uncertainty.as_image()

fig, ax = plt.subplots(1, 2, figsize=[20, 9], sharex=True, sharey=True)

# for val in pot_steps:
ax[0].contour(grid.grid[0], grid.grid[1], pred_potential,
           levels=pot_steps, linewidths=0.5, alpha=0.75,
           cmap=cm.roma, vmin=-c_amp, vmax=c_amp)
ax[0].contour(grid.grid[0], grid.grid[1], pred_potential,
           levels=[0], colors="k", linewidths=1.5)


im = ax[1].imshow(uncertainty, extent=(0, 100, 0, 100), origin='lower',
                  cmap=cm.nuuk, vmin=0)
ax[1].contour(grid.grid[0], grid.grid[1], pred_potential,
           levels=[0], colors="k", linewidths=1.5, alpha=0.5)

for a in ax:
    a.scatter(points.coordinates[:, 0], points.coordinates[:, 1],
               c=points.variables["rock_num"].measurements.values,
               cmap=cm.roma, vmin=-1, vmax=1)

    for i in range(tangents.n_data):
        a.arrow(
            tangents.coordinates[i, 0], tangents.coordinates[i, 1],
            tangents.directions[i, 0]*2, tangents.directions[i, 1]*2
        )
        a.arrow(
            tangents.coordinates[i, 0], tangents.coordinates[i, 1],
            -tangents.directions[i, 0]*2, -tangents.directions[i, 1]*2
        )

    a.set_aspect("equal")
    a.set_xlabel("X")
    a.set_ylabel("Y")

ax[0].set_title('Indicator for category "a"')
ax[1].set_title('Uncertainty')

plt.colorbar(im, ax=ax, shrink=0.8)
fig.show()

## References

Lajaunie, C., Courrioux, G., & Manuel, L. (1997). Foliation fields and 3D cartography in geology: Principles of a method based on potential interpolation. Mathematical Geology, 29(4), 571–584. https://doi.org/10.1007/BF02775087

Gonçalves, Í. G., Kumaira, S., & Guadagnin, F. (2017). A machine learning approach to the potential-field method for implicit modeling of geological structures. Computers & Geosciences, 103(March 2017), 173–182. https://doi.org/10.1016/j.cageo.2017.03.015